# QOU hecke-engine in SageMath

Interactive walkthrough of the canonical H_3(q) surface at the substrate q_0,
with cross-checks against Sage's symmetric-quantum-integer infrastructure.

**Prerequisites:** `sage -pip install pyhecke`

**Launch:** `sage -n jupyter notebooks/hecke_surface_demo.ipynb`

## Convention note

`pyhecke` uses the **symmetric** quantum integer
$[n]_q = (q^n - q^{-n})/(q - q^{-1})$ and the Markov-Ocneanu-Wenzl trace
with parameter $z = 1/(q^{1/2} + q^{-1/2})$. Sage's defaults
(`q_int`, `IwahoriHeckeAlgebra(W,q).T()`) use different conventions; we
build the symmetric form explicitly so cross-checks are apples-to-apples.

## 1. Substrate and Markov-trace parameter

In [ ]:
from pyhecke import q_dimension, wedderburn_weight, partitions_of, TR_M
from pyhecke.gram import G as gram_matrix_at_q0, nf_tr

try:
    from qou_substrate.constants import Q as Q0
except ImportError:
    from q_parameter import Q as Q0

z = 1.0 / (Q0**0.5 + (1.0/Q0)**0.5)   # symmetric MOW parameter
h = Q0 - 1.0/Q0
print(f'q_0           = {float(Q0)}')
print(f'Markov z(q_0) = {float(z):.12f}')
print(f'Hecke  h(q_0) = {float(h):.12f}')

## 2. Markov-trace weights on the 6-element NF basis

`TR_M[i]` = canonical Markov-trace weight on the i-th NF basis element.
The vector is $[1, z, z, z^2, z^2, h z^2 + z]$ — entry 5 is the longest
element $\sigma_0 \sigma_1 \sigma_0$, which by cyclicity + Hecke
relation evaluates to $h z^2 + z$, not $z^3$
(see `docs/audits/2026-05-19-trm5-markov-bug.md`).

In [ ]:
expected = [1.0, z, z, z*z, z*z, h*z*z + z]
for i, (w, e) in enumerate(zip(TR_M, expected)):
    print(f'TR_M[{i}] = {float(w):.12f}   formula {float(e):.12f}   diff {abs(float(w)-float(e)):.2e}')

## 3. Gram matrix G(q_0)

G is the 6×6 inner-product matrix of the NF basis under the Markov-trace
pairing. At q_0 it is **indefinite** (sign(det G) is not constrained).

In [ ]:
import numpy as np
G = np.array(gram_matrix_at_q0)
print('G(q_0) =')
print(G)
print()
print('det G   =', np.linalg.det(G))
print('eigvals =', sorted(np.linalg.eigvalsh((G + G.T) / 2)))
print('cond    =', np.linalg.cond(G))

## 4. Sage's IwahoriHeckeAlgebra(A_2, q)

Sage's `IwahoriHeckeAlgebra(W, q).T()` uses the Lusztig convention
$(T_i - q)(T_i + 1) = 0$. Its dimension matches QOU's NF basis size
($|S_3| = 6$); a tight numerical bridge requires the rescaling
$\sigma_i = q^{-1/2} T_i$. See `scripts/sage_iwahori_adjacency.sage`.

In [ ]:
R.<q_sym> = LaurentPolynomialRing(QQ)
W = WeylGroup(['A', 2], prefix='s')
H = IwahoriHeckeAlgebra(W, q_sym).T()
print(H)
print('dim =', H.dimension(), '   matches len(TR_M) =', len(TR_M))

## 5. q-dimension cross-check (symmetric convention)

Both sides evaluate the **same** symbolic formula
$\dim_q(\lambda) = [n]_q^{\rm sym}! \;/\; \prod_{(i,j)} [h(i,j)]_q^{\rm sym}$
at $q = q_0$; agreement is to machine precision.

In [ ]:
def q_int_sym(n):
    if n == 0:
        return R.zero()
    return sum(q_sym^(n - 1 - 2*k) for k in range(n))

def q_factorial_sym(n):
    f = R.one()
    for k in range(1, n + 1):
        f *= q_int_sym(k)
    return f

def sage_q_dim_sym(lam):
    P = Partition(list(lam))
    n = sum(P)
    if n == 0:
        return R.one()
    hp = R.one()
    for (i, j) in P.cells():
        hp *= q_int_sym(P.hook_length(i, j))
    return q_factorial_sym(n) / hp

for n in (1, 2, 3, 4, 5):
    for lam in partitions_of(n):
        s = float(sage_q_dim_sym(lam).subs(q_sym=Q0))
        p = float(q_dimension(lam))
        print(f'λ = {str(lam):<10} sage {s:>16.9f}  pyhecke {p:>16.9f}  Δ {abs(s-p):.2e}')

## 6. Wedderburn coefficients y_λ

$y_\lambda$ is the Frobenius coefficient in $\mathrm{tr}_M = \sum_\lambda y_\lambda \chi^\lambda$.
Sage gives the ordinary $\dim_{\mathbb C}({\rm Specht}_\lambda)$ for each $\lambda$;
pyhecke gives the q-deformed $y_\lambda$ evaluated at the substrate.

In [ ]:
for lam in partitions_of(3):
    P = Partition(list(lam))
    print(f'λ = {lam}   dim_C(Specht) = {P.dimension()}   y_λ(q_0) = {float(wedderburn_weight(lam)):.9f}')